# Seven-channel proposed-model source, seed 42

NDRE source variant. The external run_ndre_confirmation.py driver referenced in original cell 41 is not included in the two source notebooks.

Source: `graindiscoloration (2).ipynb`, zero-based cell 36.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
GD-RGBMS-DB V3.1 — AUXILIARY-SUPERVISED GATED FUSION
======================================================

Proposed architecture for the dataset-provided 280/60/60 benchmark:
  RGB (3 channels) and spectral (MS+NDRE, 4 channels) use separate
  encoder branches. Learned two-way soft gates fuse the modalities at
  four encoder scales. Fused information is progressively returned to
  both branches and used as decoder skips.

Training-only modality supervision:
  * a 1x1 RGB auxiliary head supervises the deepest RGB feature
  * a 1x1 spectral auxiliary head supervises the deepest MS+NDRE feature
  * the spectral auxiliary loss receives a moderately larger weight
  * both auxiliary weights decay after the first 40% of requested epochs
  * the checkpoint is selected only by the main fused validation Dice

This is a genuine one-run, from-scratch experiment. No RGB-only or
MS+NDRE-only pretrained checkpoint is loaded. Auxiliary heads are not executed
during normal validation, controlled TTA, gate analysis or final inference.

Speed choices:
  * branch base width 28 (parameter budget near the V2 baseline)
  * AMP, RAM cache for train+validation, max 70 epochs, patience 12
  * full 896x672 training; batch size 2; GroupNorm

After the best checkpoint is loaded, three prediction views are computed
once per image: original, horizontal flip and vertical flip. They are then
reused to compare four inference modes without redundant model passes:

  1. no_tta
  2. horizontal_tta
  3. vertical_tta
  4. horizontal_vertical_tta

Every mode receives a threshold selected only from validation. The primary
final mode is also selected only by validation Dice; test results never choose
the mode or its threshold.
"""

from __future__ import annotations

import csv
import json
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image
# Plain tqdm avoids Kaggle's occasional "Error displaying widget" message.
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# ============================================================================
# 1. CONFIGURATION
# ============================================================================

# Set manually only if automatic discovery fails.
# BASE_DIR = "/kaggle/input/datasets/hemal7/gd-rgbms-db"
# BASE_DIR = "/content/GD-RGBMS-DB"
BASE_DIR = None

MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

WORK_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path("/content")
)
OUT_DIR = WORK_DIR / "gd_v3_1_auxsupervised_gatedfusion_groupnorm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TARGET_SIZE = (896, 672)  # width, height
BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
MAX_EPOCHS = 70
PATIENCE = 12
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

BRANCH_BASE = 28
PREFERRED_GN_GROUPS = 8
MODALITY_DROPOUT_PROBABILITY = 0.0  # later ablation, keep 0 now

# Training-only modality-specific supervision. These weights are intentionally
# modest: the main fused output always remains the dominant objective.
AUX_RGB_INITIAL_WEIGHT = 0.15
AUX_SPECTRAL_INITIAL_WEIGHT = 0.25
AUX_RGB_MIDDLE_WEIGHT = 0.10
AUX_SPECTRAL_MIDDLE_WEIGHT = 0.15
AUX_RGB_FINAL_WEIGHT = 0.05
AUX_SPECTRAL_FINAL_WEIGHT = 0.08
AUX_MIDDLE_PHASE_START = 0.40
AUX_FINAL_PHASE_START = 0.70

# Preprocessing is expensive. Float16 RAM cache is about 2.9 GB for
# train+validation and normally saves substantial time after epoch 1.
CACHE_TRAIN_AND_VALIDATION = True
SAVE_TEST_OUTPUTS = True
SAVE_GATE_STATISTICS = True
SHOW_DOWNLOAD_LINK = True

MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)

# Same train-only statistics as the controlled V2.2 provided-split runs.
MEAN = np.array([
    0.41819797, 0.46573568, 0.25167707,
    0.21822916, 0.16206901, 0.23585168, 0.59974279,
], dtype=np.float32)
STD = np.array([
    0.15404519, 0.15691774, 0.13375558,
    0.07147331, 0.05124885, 0.04640943, 0.06006572,
], dtype=np.float32)

TRAIN_THRESHOLDS = np.round(np.arange(0.20, 0.86, 0.05), 2).tolist()
FINAL_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST


# ============================================================================
# 2. DATASET DISCOVERY AND REPRODUCIBILITY
# ============================================================================

def read_list(path):
    with open(path, "r", encoding="utf-8") as file:
        return [line.strip() for line in file if line.strip()]


def valid_root(path):
    required = [
        path / "RGB", path / "MS", path / "Mark",
        path / "train_list.txt", path / "val_list.txt", path / "test_list.txt",
    ]
    if not all(item.exists() for item in required):
        return False
    return (
        len(read_list(path / "train_list.txt")) == 280
        and len(read_list(path / "val_list.txt")) == 60
        and len(read_list(path / "test_list.txt")) == 60
    )


def find_dataset_root():
    roots = [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]
    candidates = []
    for root in roots:
        if root.exists():
            for item in root.rglob("train_list.txt"):
                if valid_root(item.parent):
                    candidates.append(item.parent)
    if not candidates:
        raise FileNotFoundError(
            "Provided 280/60/60 split not found. Set BASE_DIR manually."
        )
    candidates.sort(
        key=lambda p: ("gd-rgbms" not in str(p).lower(), len(str(p)))
    )
    return candidates[0]


if BASE_DIR is None:
    BASE_DIR = find_dataset_root()
BASE_DIR = Path(BASE_DIR)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

train_names = read_list(BASE_DIR / "train_list.txt")
val_names = read_list(BASE_DIR / "val_list.txt")
test_names = read_list(BASE_DIR / "test_list.txt")

sets = [set(train_names), set(val_names), set(test_names)]
if sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]:
    raise ValueError("Exact filename overlap exists across splits.")

rgb_dir, ms_dir, mark_dir = (
    BASE_DIR / "RGB", BASE_DIR / "MS", BASE_DIR / "Mark"
)

print("=" * 78)
print("V3.1 AUXILIARY-SUPERVISED GATED FUSION — GROUPNORM")
print("=" * 78)
print("Dataset :", BASE_DIR)
print("Output  :", OUT_DIR)
print("Device  :", device)
print("Split   :", len(train_names), len(val_names), len(test_names))
print("Input   :", TARGET_SIZE, "| batch:", BATCH_SIZE)
print("Branch base:", BRANCH_BASE, "| RAM cache:", CACHE_TRAIN_AND_VALIDATION)


# ============================================================================
# 3. PREPROCESSING AND OPTIONAL RAM CACHE
# ============================================================================

def find_path(directory, stem, extensions):
    for extension in extensions:
        path = directory / f"{stem}{extension}"
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {stem} in {directory}")


def binary_mask(image):
    array = np.asarray(image.convert("RGB"), dtype=np.int16)
    return (
        np.abs(array - MASK_POSITIVE_RGB).sum(axis=2) < 80
    ).astype(np.float32)


def compute_ndre(ms):
    rededge, nir = ms[:, :, 1], ms[:, :, 2]
    raw = (nir - rededge) / (nir + rededge + 1e-6)
    return np.clip((raw + 1.0) / 2.0, 0.0, 1.0).astype(np.float32)


def resize_float(channel):
    image = Image.fromarray(channel.astype(np.float32))
    image = image.resize(TARGET_SIZE, BILINEAR)
    return np.asarray(image, dtype=np.float32).copy()


def load_raw(name):
    stem = Path(name).stem
    rgb_path = find_path(rgb_dir, stem, [".JPG", ".jpg", ".JPEG", ".jpeg"])
    ms_path = find_path(ms_dir, stem, [".PNG", ".png"])
    mark_path = find_path(mark_dir, stem, [".png", ".PNG"])

    with Image.open(rgb_path) as item:
        rgb_image = item.convert("RGB")
    with Image.open(ms_path) as item:
        ms_image = item.convert("RGB")
    with Image.open(mark_path) as item:
        mark_image = item.convert("RGB")

    ms_original = np.asarray(ms_image, dtype=np.float32) / 255.0
    ndre_original = compute_ndre(ms_original)

    rgb_image = rgb_image.resize(TARGET_SIZE, BILINEAR)
    ms_image = ms_image.resize(TARGET_SIZE, BILINEAR)
    mark_image = mark_image.resize(TARGET_SIZE, NEAREST)

    rgb = np.asarray(rgb_image, dtype=np.float32).copy() / 255.0
    ms = np.asarray(ms_image, dtype=np.float32).copy() / 255.0
    ndre = resize_float(ndre_original)
    mask = binary_mask(mark_image)
    image = np.concatenate([rgb, ms, ndre[..., None]], axis=2)
    return image.astype(np.float32), mask, stem


RAM_CACHE = {}


def preload_cache(names):
    if not CACHE_TRAIN_AND_VALIDATION:
        return
    for name in tqdm(names, desc="Preloading train+val RAM cache"):
        image, mask, stem = load_raw(name)
        RAM_CACHE[name] = (
            image.astype(np.float16),
            mask.astype(np.uint8),
            stem,
        )
    approximate_gb = sum(
        image.nbytes + mask.nbytes
        for image, mask, _ in RAM_CACHE.values()
    ) / (1024 ** 3)
    print(f"RAM cache: {len(RAM_CACHE)} images, {approximate_gb:.2f} GB")


preload_cache(train_names + val_names)


class GDDataset(Dataset):
    def __init__(self, names, augment=False):
        self.names = list(names)
        self.augment = augment

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        if name in RAM_CACHE:
            cached_image, cached_mask, stem = RAM_CACHE[name]
            image = cached_image.astype(np.float32)
            mask = cached_mask.astype(np.float32)
        else:
            image, mask, stem = load_raw(name)

        if self.augment:
            if random.random() < 0.5:
                image, mask = np.flip(image, 1).copy(), np.flip(mask, 1).copy()
            if random.random() < 0.5:
                image, mask = np.flip(image, 0).copy(), np.flip(mask, 0).copy()
            if random.random() < 0.75:
                gain = np.random.uniform(0.88, 1.12, (1, 1, 3)).astype(np.float32)
                bias = np.random.uniform(-0.035, 0.035, (1, 1, 3)).astype(np.float32)
                image[..., :3] = np.clip(
                    image[..., :3] * gain + bias, 0.0, 1.0
                )

        image = ((image - MEAN) / STD).astype(np.float32)
        return (
            torch.from_numpy(image.transpose(2, 0, 1)).float(),
            torch.from_numpy(mask[None]).float(),
            stem,
        )


train_loader = DataLoader(
    GDDataset(train_names, augment=True),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, drop_last=False,
)
val_loader = DataLoader(
    GDDataset(val_names), batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory,
)
test_loader = DataLoader(
    GDDataset(test_names), batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory,
)


# ============================================================================
# 4. LIGHTWEIGHT MULTI-SCALE GATED MODEL
# ============================================================================

def group_norm(channels):
    groups = min(PREFERRED_GN_GROUPS, channels)
    while channels % groups:
        groups -= 1
    return nn.GroupNorm(groups, channels, eps=1e-5, affine=True)


class SCSE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.channel = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )
        self.spatial = nn.Sequential(nn.Conv2d(channels, 1, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.channel(x) + x * self.spatial(x)


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout) if dropout else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
        )
        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                group_norm(out_channels),
            )
        )
        self.attention = SCSE(out_channels)

    def forward(self, x):
        return self.attention(
            F.silu(self.block(x) + self.skip(x), inplace=True)
        )


class SoftGatedFusion(nn.Module):
    """Two learned weights sum to one at every spatial location."""

    def __init__(self, channels):
        super().__init__()
        hidden = max(channels // 8, 8)
        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, hidden, 1, bias=False),
            group_norm(hidden),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, 2, 1),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            group_norm(channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, rgb_feature, spectral_feature):
        weights = torch.softmax(
            self.gate(torch.cat([rgb_feature, spectral_feature], dim=1)),
            dim=1,
        )
        fused = (
            weights[:, 0:1] * rgb_feature
            + weights[:, 1:2] * spectral_feature
        )
        return self.refine(fused), weights


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv = ResBlock(
            in_channels + skip_channels, out_channels, dropout
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:], mode="bilinear", align_corners=False
        )
        return self.conv(torch.cat([x, skip], dim=1))


class AuxiliarySupervisedGatedUNet(nn.Module):
    def __init__(self, base=28):
        super().__init__()
        channels = [base, base * 2, base * 4, base * 8]
        c1, c2, c3, c4 = channels
        self.pool = nn.MaxPool2d(2)

        self.rgb1, self.spec1 = ResBlock(3, c1), ResBlock(4, c1)
        self.rgb2, self.spec2 = ResBlock(c1, c2), ResBlock(c1, c2)
        self.rgb3, self.spec3 = ResBlock(c2, c3, 0.05), ResBlock(c2, c3, 0.05)
        self.rgb4, self.spec4 = ResBlock(c3, c4, 0.10), ResBlock(c3, c4, 0.10)

        self.fuse1 = SoftGatedFusion(c1)
        self.fuse2 = SoftGatedFusion(c2)
        self.fuse3 = SoftGatedFusion(c3)
        self.fuse4 = SoftGatedFusion(c4)

        # Training-only heads. Their outputs are requested explicitly by
        # return_aux=True; ordinary inference does not execute them.
        self.rgb_auxiliary_head = nn.Conv2d(c4, 1, kernel_size=1)
        self.spectral_auxiliary_head = nn.Conv2d(c4, 1, kernel_size=1)

        self.center = ResBlock(c4, c4 * 2, 0.10)
        self.up4 = UpBlock(c4 * 2, c4, c4, 0.10)
        self.up3 = UpBlock(c4, c3, c3, 0.05)
        self.up2 = UpBlock(c3, c2, c2)
        self.up1 = UpBlock(c2, c1, c1)
        self.output = nn.Conv2d(c1, 1, 1)

    def modality_dropout(self, rgb, spectral):
        if not self.training or MODALITY_DROPOUT_PROBABILITY <= 0:
            return rgb, spectral
        draw = random.random()
        if draw < MODALITY_DROPOUT_PROBABILITY / 2:
            rgb = torch.zeros_like(rgb)
        elif draw < MODALITY_DROPOUT_PROBABILITY:
            spectral = torch.zeros_like(spectral)
        return rgb, spectral

    def forward(self, x, return_gates=False, return_aux=False):
        input_size = x.shape[-2:]
        rgb, spectral = self.modality_dropout(x[:, :3], x[:, 3:])

        r1, s1 = self.rgb1(rgb), self.spec1(spectral)
        f1, g1 = self.fuse1(r1, s1)

        r2 = self.rgb2(self.pool(r1 + f1))
        s2 = self.spec2(self.pool(s1 + f1))
        f2, g2 = self.fuse2(r2, s2)

        r3 = self.rgb3(self.pool(r2 + f2))
        s3 = self.spec3(self.pool(s2 + f2))
        f3, g3 = self.fuse3(r3, s3)

        r4 = self.rgb4(self.pool(r3 + f3))
        s4 = self.spec4(self.pool(s3 + f3))
        f4, g4 = self.fuse4(r4, s4)

        center = self.center(self.pool(f4))
        d4 = self.up4(center, f4)
        d3 = self.up3(d4, f3)
        d2 = self.up2(d3, f2)
        d1 = self.up1(d2, f1)
        logits = self.output(d1)

        gates = [g1, g2, g3, g4]
        if return_aux:
            rgb_auxiliary_logits = F.interpolate(
                self.rgb_auxiliary_head(r4),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )
            spectral_auxiliary_logits = F.interpolate(
                self.spectral_auxiliary_head(s4),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )
            if return_gates:
                return (
                    logits,
                    gates,
                    rgb_auxiliary_logits,
                    spectral_auxiliary_logits,
                )
            return logits, rgb_auxiliary_logits, spectral_auxiliary_logits

        return (logits, gates) if return_gates else logits


model = AuxiliarySupervisedGatedUNet(BRANCH_BASE).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
auxiliary_parameter_count = sum(
    parameter.numel()
    for name, parameter in model.named_parameters()
    if "auxiliary_head" in name
)
deployment_parameter_count = parameter_count - auxiliary_parameter_count

EXPECTED_TOTAL_PARAMETERS = 7_950_763
EXPECTED_DEPLOYMENT_PARAMETERS = 7_950_313
if (
    parameter_count != EXPECTED_TOTAL_PARAMETERS
    or deployment_parameter_count != EXPECTED_DEPLOYMENT_PARAMETERS
):
    raise RuntimeError(
        "Architecture mismatch: parameter count differs from the controlled "
        "V3.1 design."
    )

print("Training parameters  :", f"{parameter_count:,}")
print("Auxiliary parameters :", f"{auxiliary_parameter_count:,}")
print("Deployment parameters:", f"{deployment_parameter_count:,}")


# ============================================================================
# 5. LOSS, METRICS, TRAINING AND VALIDATION
# ============================================================================

class ComboLoss(nn.Module):
    def forward(self, logits, targets):
        bce_map = F.binary_cross_entropy_with_logits(
            logits, targets.float(), reduction="none"
        )
        bce = bce_map.mean()
        probabilities = torch.sigmoid(logits)
        pt = (
            probabilities * targets
            + (1 - probabilities) * (1 - targets)
        )
        focal = (bce_map * ((1 - pt) ** 2.0)).mean()
        dims = tuple(range(1, logits.ndim))
        intersection = (probabilities * targets).sum(dims)
        denominator = probabilities.sum(dims) + targets.sum(dims)
        dice = 1 - (
            (2 * intersection + 1e-6) / (denominator + 1e-6)
        ).mean()
        return 0.35 * bce + 0.25 * focal + 0.40 * dice


criterion = ComboLoss()


def confusion(probabilities, masks, threshold):
    prediction, target = probabilities >= threshold, masks >= 0.5
    return [
        int((prediction & target).sum().item()),
        int((prediction & ~target).sum().item()),
        int((~prediction & target).sum().item()),
        int((~prediction & ~target).sum().item()),
    ]


def metrics(values):
    tp, fp, fn, tn = values
    e = 1e-7
    return {
        "dice": (2 * tp + e) / (2 * tp + fp + fn + e),
        "iou": (tp + e) / (tp + fp + fn + e),
        "precision": (tp + e) / (tp + fp + e),
        "recall": (tp + e) / (tp + fn + e),
        "accuracy": (tp + tn + e) / (tp + fp + fn + tn + e),
    }


optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=4, min_lr=1e-6
)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")


def auxiliary_loss_weights(epoch):
    """Use an interpretable three-phase auxiliary-loss schedule."""
    progress = epoch / max(MAX_EPOCHS, 1)
    if progress <= AUX_MIDDLE_PHASE_START:
        return AUX_RGB_INITIAL_WEIGHT, AUX_SPECTRAL_INITIAL_WEIGHT
    if progress <= AUX_FINAL_PHASE_START:
        return AUX_RGB_MIDDLE_WEIGHT, AUX_SPECTRAL_MIDDLE_WEIGHT
    return AUX_RGB_FINAL_WEIGHT, AUX_SPECTRAL_FINAL_WEIGHT


def train_epoch(epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    rgb_weight, spectral_weight = auxiliary_loss_weights(epoch)
    sums = {
        "total": 0.0,
        "main": 0.0,
        "rgb_auxiliary": 0.0,
        "spectral_auxiliary": 0.0,
    }

    for step, (images, masks, _) in enumerate(
        tqdm(train_loader, desc="Training", leave=False), start=1
    ):
        images, masks = (
            images.to(device, non_blocking=True),
            masks.to(device, non_blocking=True),
        )
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            (
                main_logits,
                rgb_auxiliary_logits,
                spectral_auxiliary_logits,
            ) = model(images, return_aux=True)

            main_loss = criterion(main_logits, masks)
            rgb_auxiliary_loss = criterion(rgb_auxiliary_logits, masks)
            spectral_auxiliary_loss = criterion(
                spectral_auxiliary_logits,
                masks,
            )
            total_loss = (
                main_loss
                + rgb_weight * rgb_auxiliary_loss
                + spectral_weight * spectral_auxiliary_loss
            )
            scaled_loss = total_loss / ACCUMULATION_STEPS

        scaler.scale(scaled_loss).backward()
        sums["total"] += total_loss.item()
        sums["main"] += main_loss.item()
        sums["rgb_auxiliary"] += rgb_auxiliary_loss.item()
        sums["spectral_auxiliary"] += spectral_auxiliary_loss.item()

        if step % ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

    return {
        key: value / len(train_loader)
        for key, value in sums.items()
    } | {
        "rgb_weight": rgb_weight,
        "spectral_weight": spectral_weight,
    }


@torch.no_grad()
def validate():
    model.eval()
    totals = {threshold: [0, 0, 0, 0] for threshold in TRAIN_THRESHOLDS}
    loss_total = 0.0
    for images, masks, _ in tqdm(val_loader, desc="Validation", leave=False):
        images, masks = images.to(device), masks.to(device)
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            logits = model(images)
            loss_total += criterion(logits, masks).item()
        probabilities = torch.sigmoid(logits)
        for threshold in TRAIN_THRESHOLDS:
            values = confusion(probabilities, masks, threshold)
            totals[threshold] = [
                old + new for old, new in zip(totals[threshold], values)
            ]
    rows = []
    for threshold, values in totals.items():
        row = metrics(values)
        row.update({
            "threshold": float(threshold),
            "loss": loss_total / len(val_loader),
        })
        rows.append(row)
    return max(rows, key=lambda item: item["dice"])


checkpoint_path = (
    OUT_DIR / "best_v3_1_auxsupervised_gatedfusion_groupnorm.pt"
)
history = []
best_dice, best_epoch = -1.0, 0

for epoch in range(1, MAX_EPOCHS + 1):
    start = time.time()
    train = train_epoch(epoch)
    val = validate()
    scheduler.step(val["dice"])
    minutes = (time.time() - start) / 60
    lr = optimizer.param_groups[0]["lr"]
    row = {
        "epoch": epoch,
        "train_total_loss": train["total"],
        "train_main_loss": train["main"],
        "train_rgb_auxiliary_loss": train["rgb_auxiliary"],
        "train_spectral_auxiliary_loss": train["spectral_auxiliary"],
        "rgb_auxiliary_weight": train["rgb_weight"],
        "spectral_auxiliary_weight": train["spectral_weight"],
        "val_main_loss": val["loss"],
        "val_dice": val["dice"], "val_iou": val["iou"],
        "val_precision": val["precision"], "val_recall": val["recall"],
        "val_accuracy": val["accuracy"], "val_threshold": val["threshold"],
        "learning_rate": lr, "minutes": minutes,
    }
    history.append(row)
    print(
        f"Epoch {epoch:03d} | "
        f"loss total/main {train['total']:.4f}/{train['main']:.4f} | "
        f"aux RGB/MS {train['rgb_auxiliary']:.4f}/"
        f"{train['spectral_auxiliary']:.4f} | "
        f"w {train['rgb_weight']:.3f}/{train['spectral_weight']:.3f} | "
        f"val loss {val['loss']:.4f} | "
        f"Dice {val['dice']:.4f} | IoU {val['iou']:.4f} | "
        f"P {val['precision']:.4f} | R {val['recall']:.4f} | "
        f"th {val['threshold']:.2f} | lr {lr:.2e} | {minutes:.1f}m"
    )
    if val["dice"] > best_dice:
        best_dice, best_epoch = val["dice"], epoch
        torch.save({
            "model": model.state_dict(), "epoch": epoch,
            "best_non_tta_val_dice": best_dice,
            "training_val_threshold": val["threshold"],
            "target_size": TARGET_SIZE, "mean": MEAN.tolist(),
            "std": STD.tolist(), "seed": SEED,
            "architecture": "AuxiliarySupervisedGatedUNet",
            "branch_base": BRANCH_BASE,
            "normalization": "GroupNorm",
            "parameter_count": parameter_count,
            "deployment_parameter_count": deployment_parameter_count,
            "auxiliary_parameter_count": auxiliary_parameter_count,
            "auxiliary_supervision": {
                "rgb_initial_weight": AUX_RGB_INITIAL_WEIGHT,
                "spectral_initial_weight": AUX_SPECTRAL_INITIAL_WEIGHT,
                "rgb_middle_weight": AUX_RGB_MIDDLE_WEIGHT,
                "spectral_middle_weight": AUX_SPECTRAL_MIDDLE_WEIGHT,
                "rgb_final_weight": AUX_RGB_FINAL_WEIGHT,
                "spectral_final_weight": AUX_SPECTRAL_FINAL_WEIGHT,
                "middle_phase_start": AUX_MIDDLE_PHASE_START,
                "final_phase_start": AUX_FINAL_PHASE_START,
                "weights_at_best_epoch": {
                    "rgb": train["rgb_weight"],
                    "spectral": train["spectral_weight"],
                },
                "checkpoint_selected_by": "main fused validation Dice only",
            },
        }, checkpoint_path)
        print("Saved:", checkpoint_path)
    if epoch - best_epoch >= PATIENCE:
        print(f"Early stopping after {PATIENCE} non-improving epochs.")
        break


def save_csv(path, rows):
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


history_path = OUT_DIR / "training_history.csv"
save_csv(history_path, history)


# ============================================================================
# 6. CONTROLLED TTA ABLATION, GATE ANALYSIS AND FINAL TEST
# ============================================================================

def safe_load(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


checkpoint = safe_load(checkpoint_path)
model.load_state_dict(checkpoint["model"], strict=True)
model.eval()

# Each view is inferred only once. Four ablation modes are constructed later.
TTA_MODES = (
    "no_tta",
    "horizontal_tta",
    "vertical_tta",
    "horizontal_vertical_tta",
)


@torch.no_grad()
def predict_views(images):
    """Return aligned original, horizontal-flip and vertical-flip predictions."""
    original = torch.sigmoid(model(images))
    horizontal = torch.flip(
        torch.sigmoid(model(torch.flip(images, dims=[-1]))),
        dims=[-1],
    )
    vertical = torch.flip(
        torch.sigmoid(model(torch.flip(images, dims=[-2]))),
        dims=[-2],
    )
    return {
        "original": original,
        "horizontal": horizontal,
        "vertical": vertical,
    }


@torch.no_grad()
def collect_views(loader, description):
    """Store float32 views so threshold comparisons remain fully precise."""
    records = []
    for images, masks, stems in tqdm(loader, desc=description):
        views = predict_views(images.to(device, non_blocking=True))
        records.append({
            "stem": stems[0],
            "mask": masks[0, 0].numpy().astype(np.float32),
            "original": (
                views["original"][0, 0].cpu().numpy().astype(np.float32)
            ),
            "horizontal": (
                views["horizontal"][0, 0].cpu().numpy().astype(np.float32)
            ),
            "vertical": (
                views["vertical"][0, 0].cpu().numpy().astype(np.float32)
            ),
        })
    return records


def aggregate_probability(view_record, mode):
    """Create one float32 probability map for the requested ablation mode."""
    if mode == "no_tta":
        selected = [view_record["original"]]
    elif mode == "horizontal_tta":
        selected = [
            view_record["original"],
            view_record["horizontal"],
        ]
    elif mode == "vertical_tta":
        selected = [
            view_record["original"],
            view_record["vertical"],
        ]
    elif mode == "horizontal_vertical_tta":
        selected = [
            view_record["original"],
            view_record["horizontal"],
            view_record["vertical"],
        ]
    else:
        raise ValueError(f"Unknown TTA mode: {mode}")
    return np.mean(
        np.stack(selected, axis=0),
        axis=0,
        dtype=np.float32,
    )


def records_for_mode(view_records, mode):
    return [
        {
            "stem": record["stem"],
            "mask": record["mask"],
            "probability": aggregate_probability(record, mode),
        }
        for record in view_records
    ]


def threshold_search(records, thresholds):
    rows = []
    for threshold in thresholds:
        total = [0, 0, 0, 0]
        for record in records:
            values = confusion(
                record["probability"],
                record["mask"],
                threshold,
            )
            total = [old + new for old, new in zip(total, values)]
        row = metrics(total)
        row["threshold"] = float(threshold)
        rows.append(row)
    return max(rows, key=lambda item: item["dice"]), rows


def evaluate_records(records, threshold, mode):
    global_values = [0, 0, 0, 0]
    per_image = []
    for record in records:
        values = confusion(
            record["probability"],
            record["mask"],
            threshold,
        )
        global_values = [
            old + new for old, new in zip(global_values, values)
        ]
        row = metrics(values)
        target = record["mask"] >= 0.5
        prediction = record["probability"] >= threshold
        row.update({
            "stem": record["stem"],
            "tta_mode": mode,
            "threshold": threshold,
            "gt_coverage_percent": float(target.mean() * 100),
            "predicted_coverage_percent": float(prediction.mean() * 100),
            "soft_coverage_percent": float(
                record["probability"].mean() * 100
            ),
            "tp": values[0],
            "fp": values[1],
            "fn": values[2],
            "tn": values[3],
        })
        per_image.append(row)
    result = metrics(global_values)
    result.update({
        "tta_mode": mode,
        "threshold": threshold,
        "tp": global_values[0],
        "fp": global_values[1],
        "fn": global_values[2],
        "tn": global_values[3],
    })
    return result, per_image


# --------------------------------------------------------------------------
# 6A. VALIDATION: tune one threshold per mode, then select the primary mode.
# Selection is based only on validation Dice.
# --------------------------------------------------------------------------

validation_views = collect_views(
    val_loader,
    "Validation views for TTA ablation",
)
validation_results = {}
validation_threshold_paths = {}

print("\n" + "=" * 78)
print("VALIDATION TTA ABLATION")
print("=" * 78)

for mode in TTA_MODES:
    mode_records = records_for_mode(validation_views, mode)
    val_best, threshold_rows = threshold_search(
        mode_records,
        FINAL_THRESHOLDS,
    )
    val_best = {"tta_mode": mode, **val_best}
    validation_results[mode] = val_best

    threshold_path = (
        OUT_DIR / f"validation_threshold_search_{mode}.csv"
    )
    threshold_rows = [
        {"tta_mode": mode, **row} for row in threshold_rows
    ]
    save_csv(threshold_path, threshold_rows)
    validation_threshold_paths[mode] = str(threshold_path)

    print(
        f"{mode:27s} | Dice={val_best['dice']:.4f} | "
        f"IoU={val_best['iou']:.4f} | "
        f"P={val_best['precision']:.4f} | "
        f"R={val_best['recall']:.4f} | "
        f"th={val_best['threshold']:.2f}"
    )
    del mode_records

selected_mode = max(
    TTA_MODES,
    key=lambda item: validation_results[item]["dice"],
)
selected_threshold = validation_results[selected_mode]["threshold"]

print("\nVALIDATION-SELECTED FINAL MODE")
print("Mode      :", selected_mode)
print("Threshold :", f"{selected_threshold:.2f}")
print(
    "Val Dice  :",
    f"{validation_results[selected_mode]['dice']:.4f}",
)

del validation_views
if device.type == "cuda":
    torch.cuda.empty_cache()


# --------------------------------------------------------------------------
# 6B. GATE STATISTICS: original orientation only.
# --------------------------------------------------------------------------

@torch.no_grad()
def gate_statistics(loader, split_name):
    rows = []
    for images, _, stems in tqdm(
        loader,
        desc=f"{split_name} gate statistics",
    ):
        _, gates = model(
            images.to(device, non_blocking=True),
            return_gates=True,
        )
        row = {"split": split_name, "stem": stems[0]}
        for scale, gate in enumerate(gates, start=1):
            row[f"scale{scale}_rgb_weight"] = float(
                gate[:, 0].mean().cpu()
            )
            row[f"scale{scale}_spectral_weight"] = float(
                gate[:, 1].mean().cpu()
            )
        rows.append(row)
    return rows


gate_rows = []
if SAVE_GATE_STATISTICS:
    gate_rows.extend(gate_statistics(val_loader, "validation"))


# --------------------------------------------------------------------------
# 6C. TEST: report every ablation mode, but never use test to choose a mode.
# --------------------------------------------------------------------------

test_views = collect_views(
    test_loader,
    "Test views for TTA ablation",
)

if SAVE_GATE_STATISTICS:
    gate_rows.extend(gate_statistics(test_loader, "test"))
    save_csv(
        OUT_DIR / "modality_gate_statistics.csv",
        gate_rows,
    )

test_results = {}
per_image_paths = {}

print("\n" + "=" * 78)
print("TEST TTA ABLATION — THRESHOLDS FIXED FROM VALIDATION")
print("=" * 78)

for mode in TTA_MODES:
    mode_records = records_for_mode(test_views, mode)
    threshold = validation_results[mode]["threshold"]
    test_result, per_image = evaluate_records(
        mode_records,
        threshold,
        mode,
    )
    test_results[mode] = test_result

    per_image_path = (
        OUT_DIR / f"per_image_test_metrics_{mode}.csv"
    )
    save_csv(per_image_path, per_image)
    per_image_paths[mode] = str(per_image_path)

    print(
        f"{mode:27s} | Dice={test_result['dice']:.4f} | "
        f"IoU={test_result['iou']:.4f} | "
        f"P={test_result['precision']:.4f} | "
        f"R={test_result['recall']:.4f} | "
        f"th={threshold:.2f}"
    )
    del mode_records, per_image

selected_test_metrics = test_results[selected_mode]

ablation_rows = []
for mode in TTA_MODES:
    val_result = validation_results[mode]
    test_result = test_results[mode]
    ablation_rows.append({
        "tta_mode": mode,
        "selected_by_validation": mode == selected_mode,
        "validation_threshold": val_result["threshold"],
        "validation_dice": val_result["dice"],
        "validation_iou": val_result["iou"],
        "validation_precision": val_result["precision"],
        "validation_recall": val_result["recall"],
        "validation_accuracy": val_result["accuracy"],
        "test_dice": test_result["dice"],
        "test_iou": test_result["iou"],
        "test_precision": test_result["precision"],
        "test_recall": test_result["recall"],
        "test_accuracy": test_result["accuracy"],
        "test_tp": test_result["tp"],
        "test_fp": test_result["fp"],
        "test_fn": test_result["fn"],
        "test_tn": test_result["tn"],
    })

ablation_summary_path = OUT_DIR / "tta_ablation_summary.csv"
save_csv(ablation_summary_path, ablation_rows)

selected_test_path = OUT_DIR / "selected_final_test_metrics.csv"
save_csv(selected_test_path, [selected_test_metrics])

primary_no_tta_path = OUT_DIR / "primary_no_tta_test_metrics.csv"
save_csv(primary_no_tta_path, [test_results["no_tta"]])

print("\n" + "=" * 78)
print("PRIMARY FINAL TEST — MODE SELECTED USING VALIDATION ONLY")
print("=" * 78)
print(json.dumps(selected_test_metrics, indent=2))


# Save masks and probability maps only for the validation-selected mode.
if SAVE_TEST_OUTPUTS:
    prediction_dir = OUT_DIR / "selected_test_predictions"
    probability_dir = OUT_DIR / "selected_test_probability_uint16"
    prediction_dir.mkdir(exist_ok=True)
    probability_dir.mkdir(exist_ok=True)
    selected_records = records_for_mode(test_views, selected_mode)
    for record in tqdm(
        selected_records,
        desc="Saving selected-mode test outputs",
    ):
        prediction = (
            record["probability"] >= selected_threshold
        ).astype(np.uint8) * 255
        probability = np.clip(
            record["probability"] * 65535,
            0,
            65535,
        ).astype(np.uint16)
        Image.fromarray(prediction).save(
            prediction_dir / f"{record['stem']}.png"
        )
        Image.fromarray(probability).save(
            probability_dir / f"{record['stem']}.png"
        )
    del selected_records

del test_views
if device.type == "cuda":
    torch.cuda.empty_cache()


# ============================================================================
# 7. CONFIGURATION, CHECKPOINT METADATA AND DOWNLOADABLE ZIP
# ============================================================================

configuration = {
    "version": (
        "V3.1 Lightweight Multi-Scale Gated Fusion GroupNorm with "
        "Training-Only Modality-Specific Auxiliary Supervision"
    ),
    "dataset_root": str(BASE_DIR),
    "seed": SEED,
    "split_sizes": {
        "train": len(train_names),
        "validation": len(val_names),
        "test": len(test_names),
    },
    "target_size_width_height": list(TARGET_SIZE),
    "batch_size": BATCH_SIZE,
    "accumulation_steps": ACCUMULATION_STEPS,
    "epochs_requested": MAX_EPOCHS,
    "epochs_completed": len(history),
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "architecture": "AuxiliarySupervisedGatedUNet",
    "training_initialization": "from_scratch_single_run",
    "external_pretrained_weights": False,
    "rgb_channels": 3,
    "spectral_channels": 4,
    "gated_encoder_scales": 4,
    "branch_base": BRANCH_BASE,
    "normalization": "GroupNorm",
    "preferred_group_count": PREFERRED_GN_GROUPS,
    "modality_dropout_probability": MODALITY_DROPOUT_PROBABILITY,
    "training_parameter_count": parameter_count,
    "auxiliary_parameter_count": auxiliary_parameter_count,
    "deployment_parameter_count": deployment_parameter_count,
    "auxiliary_supervision": {
        "heads": (
            "Training-only 1x1 segmentation heads on deepest RGB and "
            "spectral branch features"
        ),
        "rgb_initial_weight": AUX_RGB_INITIAL_WEIGHT,
        "spectral_initial_weight": AUX_SPECTRAL_INITIAL_WEIGHT,
        "rgb_middle_weight": AUX_RGB_MIDDLE_WEIGHT,
        "spectral_middle_weight": AUX_SPECTRAL_MIDDLE_WEIGHT,
        "rgb_final_weight": AUX_RGB_FINAL_WEIGHT,
        "spectral_final_weight": AUX_SPECTRAL_FINAL_WEIGHT,
        "middle_phase_start": AUX_MIDDLE_PHASE_START,
        "final_phase_start": AUX_FINAL_PHASE_START,
        "decay_type": "three_phase_piecewise_schedule",
        "target": "same binary segmentation mask as main fused output",
        "checkpoint_selection": "main fused validation Dice only",
        "executed_during_final_inference": False,
    },
    "mean": MEAN.tolist(),
    "std": STD.tolist(),
    "normalization_statistics_source": "provided training split only",
    "cache_train_validation": CACHE_TRAIN_AND_VALIDATION,
    "best_epoch": checkpoint["epoch"],
    "best_non_tta_val_dice": checkpoint["best_non_tta_val_dice"],
    "tta_modes": list(TTA_MODES),
    "tta_view_reuse": (
        "Original, horizontal and vertical predictions computed once "
        "and reused for all modes"
    ),
    "validation_results_by_mode": validation_results,
    "selected_tta_mode_source": "highest validation Dice only",
    "selected_tta_mode": selected_mode,
    "selected_threshold_source": (
        "validation threshold search for selected TTA mode"
    ),
    "selected_threshold": selected_threshold,
    "test_results_by_mode": test_results,
    "primary_no_tta_test_metrics": test_results["no_tta"],
    "primary_final_test_metrics": selected_test_metrics,
    "test_use_policy": (
        "Test metrics are reported for ablation only; test did not select "
        "the TTA mode or threshold"
    ),
}
configuration_path = OUT_DIR / "experiment_configuration.json"
with open(configuration_path, "w", encoding="utf-8") as file:
    json.dump(configuration, file, indent=2)

checkpoint["tta_validation_results"] = validation_results
checkpoint["selected_tta_mode"] = selected_mode
checkpoint["selected_threshold"] = selected_threshold
checkpoint["tta_test_results"] = test_results
checkpoint["primary_no_tta_test_metrics"] = test_results["no_tta"]
checkpoint["primary_final_test_metrics"] = selected_test_metrics
torch.save(checkpoint, checkpoint_path)

archive_path = shutil.make_archive(
    str(
        WORK_DIR
        / "gd_v3_1_auxsupervised_gatedfusion_groupnorm_results"
    ),
    "zip",
    str(OUT_DIR),
)

print("\n" + "=" * 78)
print("V3.1 AUXILIARY-SUPERVISED GATED FUSION FINISHED")
print("=" * 78)
print("Best checkpoint       :", checkpoint_path)
print("Training history      :", history_path)
print("TTA ablation summary  :", ablation_summary_path)
print("Primary no-TTA metrics:", primary_no_tta_path)
print("Selected test metrics :", selected_test_path)
print("Configuration         :", configuration_path)
print("Downloadable ZIP      :", archive_path)
print("\nIMPORTANT: download the ZIP before ending the Kaggle session.")

if SHOW_DOWNLOAD_LINK:
    try:
        from IPython.display import FileLink, display
        display(FileLink(archive_path))
    except Exception as error:
        print("Could not display an inline link:", error)